# 71 - Paired Statistical Significance Testing for Headline Comparisons

This thesis reports many per-query improvements (e.g. Recall@1000 0.654 to 0.678, or NDCG@10 0.034 to 0.086) without a formal significance test, relying instead on qualitative language ("genuine," "unambiguous," "consistently"). This notebook adds paired significance testing for the three comparisons in this thesis where the claim of one system beating another is most load-bearing: the tuned vs untuned fusion ranker, the neural network vs the tree-ensemble fusion ranker, and approximate vs exact nearest-neighbour search. All three comparisons are naturally paired (the same 19 deep-coverage queries scored by both systems being compared), which is exactly the setting a paired test is designed for, rather than an unpaired test that would ignore the shared query-level variance.

**Method.** For each comparison, two tests are reported side by side: the Wilcoxon signed-rank test (a standard, distribution-free choice for paired IR per-query score differences), and a paired bootstrap 95% confidence interval on the mean per-query difference (resampling queries with replacement, 10,000 iterations), which additionally gives an effect-size-like sense of the difference's magnitude and stability, not just whether it is distinguishable from zero. Every test uses the same 19 deep-coverage queries this thesis already treats as its leakage-free evaluation set, at NDCG@1000 (the thesis's headline broad-list metric) and NDCG@10 (to check whether the top-of-list-vs-broad-list trade-off pattern documented qualitatively throughout this thesis is also statistically real, not just directionally consistent).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

OUTPUT_DIR = Path("result/71_significance_testing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEEP_QUERY_IDS = [1, 2, 3, 4, 5, 11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]
RNG = np.random.default_rng(0)
N_BOOTSTRAP = 10_000


def paired_vectors(df_a, df_b, k, metric, query_ids=DEEP_QUERY_IDS):
    # aligns two per-query eval dataframes on query_id for a fixed k/metric, returning matched arrays
    a = df_a[(df_a["k"] == k) & (df_a["query_id"].isin(query_ids))].set_index("query_id")[metric]
    b = df_b[(df_b["k"] == k) & (df_b["query_id"].isin(query_ids))].set_index("query_id")[metric]
    common = sorted(set(a.index) & set(b.index))
    assert len(common) == len(query_ids), f"expected {len(query_ids)} matched queries, got {len(common)}"
    return a.loc[common].values, b.loc[common].values


def paired_bootstrap_ci(diffs, n_bootstrap=N_BOOTSTRAP):
    n = len(diffs)
    boot_means = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample_idx = RNG.integers(0, n, n)
        boot_means[i] = diffs[sample_idx].mean()
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return lo, hi


def run_paired_test(name, vals_a, vals_b, label_a, label_b):
    diffs = vals_b - vals_a  # positive means b > a
    mean_diff = diffs.mean()
    ci_lo, ci_hi = paired_bootstrap_ci(diffs)
    try:
        stat, p = wilcoxon(vals_a, vals_b)
    except ValueError as e:
        # all-zero differences (identical scores on every query) makes wilcoxon undefined -- report that plainly
        stat, p = np.nan, np.nan
    row = {
        "comparison": name, "system_a": label_a, "system_b": label_b,
        "mean_a": vals_a.mean(), "mean_b": vals_b.mean(), "mean_diff_b_minus_a": mean_diff,
        "bootstrap_ci_lo": ci_lo, "bootstrap_ci_hi": ci_hi,
        "wilcoxon_stat": stat, "wilcoxon_p": p,
        "significant_at_05": (p < 0.05) if not np.isnan(p) else None,
    }
    return row

In [ ]:
# --- Comparison 1: fusion ranker, tuned vs untuned GBDT (Section 4.9.9 hyperparameter tuning) ---
baseline_eval = pd.read_csv("result/61_fusion_ranker_hyperparameter_tuning/baseline_eval.csv")
tuned_eval = pd.read_csv("result/61_fusion_ranker_hyperparameter_tuning/tuned_eval.csv")

results = []
for k in [10, 1000]:
    a, b = paired_vectors(baseline_eval, tuned_eval, k, "ndcg")
    results.append(run_paired_test(f"gbdt_untuned_vs_tuned_ndcg@{k}", a, b, "untuned", "tuned"))
for k in [1000]:
    a, b = paired_vectors(baseline_eval, tuned_eval, k, "recall")
    results.append(run_paired_test(f"gbdt_untuned_vs_tuned_recall@{k}", a, b, "untuned", "tuned"))

print("=== GBDT untuned vs tuned ===")
for r in results[-3:]:
    sig = "significant" if r["significant_at_05"] else "not significant"
    print(f"  {r['comparison']}: mean {r['system_a']}={r['mean_a']:.4f}, {r['system_b']}={r['mean_b']:.4f}, "
          f"diff={r['mean_diff_b_minus_a']:+.4f} [{r['bootstrap_ci_lo']:+.4f}, {r['bootstrap_ci_hi']:+.4f}], "
          f"Wilcoxon p={r['wilcoxon_p']:.4f} ({sig})")

In [ ]:
# --- Comparison 2: neural network vs GBDT (both tuned and untuned) (Section 4.9.10) ---
neural_eval = pd.read_csv("result/64_neural_fusion_ranker/final_eval.csv")
neural_eval = neural_eval[neural_eval["system"] == "neural_net"]

for other_name, other_df in [("gbdt_untuned", baseline_eval), ("gbdt_tuned", tuned_eval)]:
    for k in [10, 1000]:
        a, b = paired_vectors(other_df, neural_eval, k, "ndcg")
        results.append(run_paired_test(f"neural_net_vs_{other_name}_ndcg@{k}", a, b, other_name, "neural_net"))

print("=== Neural network vs GBDT (untuned and tuned) ===")
for r in results[-4:]:
    sig = "significant" if r["significant_at_05"] else "not significant"
    print(f"  {r['comparison']}: mean {r['system_a']}={r['mean_a']:.4f}, {r['system_b']}={r['mean_b']:.4f}, "
          f"diff={r['mean_diff_b_minus_a']:+.4f} [{r['bootstrap_ci_lo']:+.4f}, {r['bootstrap_ci_hi']:+.4f}], "
          f"Wilcoxon p={r['wilcoxon_p']:.4f} ({sig})")

In [ ]:
# --- Comparison 3: ANN vs exact search, GTE-large, cheaper build config (M=16/efc=40/ef_search=500) at the 400K tier ---
# Matches the exact configuration reported in Table (tab:ann_tuning) of the thesis.
ann_all = pd.read_csv("result/63_ann_index_tuning/ann_results_all.csv")
exact_all = pd.read_csv("result/60_scaling_evaluation/scaling_eval_per_query.csv")

ann_400k = ann_all[(ann_all["tier"] == "400k") & (ann_all["M"] == 16) & (ann_all["ef_construction"] == 40) & (ann_all["ef_search"] == 500)]
exact_400k_gte = exact_all[(exact_all["tier"] == "400k") & (exact_all["method"] == "gte")]

for k in [1000]:
    a, b = paired_vectors(exact_400k_gte, ann_400k, k, "ndcg")
    results.append(run_paired_test(f"ann_vs_exact_ndcg@{k}_400k", a, b, "exact", "ann"))
    a, b = paired_vectors(exact_400k_gte, ann_400k, k, "recall")
    results.append(run_paired_test(f"ann_vs_exact_recall@{k}_400k", a, b, "exact", "ann"))

print("=== ANN vs exact search (GTE-large, 400K tier, M=16/efc=40/ef_search=500) ===")
for r in results[-2:]:
    sig = "significant" if r["significant_at_05"] else "not significant"
    print(f"  {r['comparison']}: mean {r['system_a']}={r['mean_a']:.4f}, {r['system_b']}={r['mean_b']:.4f}, "
          f"diff={r['mean_diff_b_minus_a']:+.4f} [{r['bootstrap_ci_lo']:+.4f}, {r['bootstrap_ci_hi']:+.4f}], "
          f"Wilcoxon p={r['wilcoxon_p']:.4f} ({sig})")

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "significance_tests.csv", index=False)
print()
print("Full results table:")
print(results_df.to_string(index=False))
print()
print(f"Saved to {OUTPUT_DIR / 'significance_tests.csv'}")